In [ ]:
import asyncio
import os
import pandas as pd
from dotenv import load_dotenv

from utils.poly_websocket import PolyOrderBookWebsocket
from utils.kalshi_websocket import KalshiOrdeBookWebsocket

In [7]:

class ArbitrageManager:
    def __init__(self):
        # Stores latest data from both platforms
        self.live_state = {}

    def check_for_arb(self, match_id):
        m = self.live_state.get(match_id)
        # Ensure we have data from both platforms before checking
        if not m or "poly" not in m or "kalshi" not in m:
            return

        p = m["poly"]
        k = m["kalshi"]

        # Strategy 1: Buy YES on Poly + Buy NO on Kalshi
        # Both prices must exist
        if p["yes_ask"] is not None and k["no_ask"] is not None:
            cost = p["yes_ask"] + k["no_ask"]
            if cost < 0.99: # Adjusted to 1% profit margin
                print(f"💰 ARB FOUND! Match: {match_id}")
                print(f"   Action: Buy Poly YES ({p['yes_ask']}) + Buy Kalshi NO ({k['no_ask']}) | Total Cost: {cost:.3f}")

        # Strategy 2: Buy YES on Kalshi + Buy NO on Poly
        if k["yes_ask"] is not None and p["no_ask"] is not None:
            cost = k["yes_ask"] + p["no_ask"]
            if cost < 0.99:
                print(f"💰 ARB FOUND! Match: {match_id}")
                print(f"   Action: Buy Kalshi YES ({k['yes_ask']}) + Buy Poly NO ({p['no_ask']}) | Total Cost: {cost:.3f}")

    async def handle_update(self, data):
        """
        This is the callback function. 
        'data' is the dictionary sent by the collectors.
        """
        m_id = data["match_id"]
        platform = data["platform"].lower()
        
        if m_id not in self.live_state:
            self.live_state[m_id] = {}
        
        # side_data contains {"YES": {"bid": X, "ask": Y}, "NO": {"bid": X, "ask": Y}}
        side_data = data["data"]
        
        # Extracting prices. Note: Our collectors already pre-calculate 'ask' where possible.
        self.live_state[m_id][platform] = {
            "yes_bid": side_data["YES"]["bid"],
            "yes_ask": side_data["YES"]["ask"],
            "no_bid": side_data["NO"]["bid"],
            "no_ask": side_data["NO"]["ask"]
        }

        # Every time a price moves, check for arbitrage
        self.check_for_arb(m_id)

# --- MAIN EXECUTION ---
async def main():
    load_dotenv()
    arb_manager = ArbitrageManager()
    
    # 1. Data Preparation
    arb_candidates = pd.read_json('data/arb_candidates.json')
    
    # Map Poly IDs to Match IDs
    p_map = {str(row['market_ids']['poly']): row['match_id'] for _, row in arb_candidates.iterrows()}
    
    # Map Kalshi Tickers to Match IDs
    k_map = {row['market_ids']['kalshi']: row['match_id'] for _, row in arb_candidates.iterrows()}

    # 2. Collector Initialization
    # We pass 'arb_manager.handle_update' as the callback (no parentheses!)
    
    poly = PolyOrderBookWebsocket(
        match_map=p_map, 
        on_update=arb_manager.handle_update
    )
    
    kalshi = KalshiOrdeBookWebsocket(
        api_id=os.environ["KALSHI_API_KEY_ID"], 
        key_path=os.environ["KALSHI_API_KEY_PATH"], 
        ticker_map=k_map, 
        on_update=arb_manager.handle_update
    )

    # 3. Running concurrently
    print("Starting Arbitrage Engine...")
    await asyncio.gather(
        poly.start(),
        kalshi.start()
    )

In [8]:
await main()

Starting Arbitrage Engine...
Fetching token IDs for 25 markets...
📡 Connected to Kalshi! Subscribing to 25 tickers.
Token map initialized.
📡 Connected to Polymarket WebSocket!


CancelledError: 